In [ ]:
import os
import xarray as xr
import geopandas as gpd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from shapely.ops import unary_union
from shapely.geometry import box
import matplotlib.dates as mdates
from matplotlib.patches import Patch

%matplotlib inline

plt.rcParams.update({
    "font.family":        "serif",
    "font.serif":         ["Times New Roman"],
    "mathtext.fontset":   "stix",
    "axes.unicode_minus": False,
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
})

# ----------------------------
# 1. Paths
# ----------------------------
nc_dir   = r"E:\tem\1961-2025tem(4h)"
shp_path = r"E:\矢量边界\gz_bountary\gz_city.shp"
out_dir  = r"E:\关中干旱小论文\研究区域图-图一"
out_pdf  = os.path.join(out_dir, "Fig2_2025_MAM_daily_anomaly_timeseries_t2m.pdf")
os.makedirs(out_dir, exist_ok=True)

# ----------------------------
# 2. 逐年读取 & 合并 & 日均
# ----------------------------
def load_year(year, nc_dir):
    fpath = os.path.join(nc_dir, f"era5_2mtemp_{year}.nc")
    ds    = xr.open_dataset(fpath, engine="h5netcdf")
    t2m   = ds["t2m"].rename({"valid_time": "time",
                               "latitude":   "lat",
                               "longitude":  "lon"})
    # 填充值处理
    fill  = t2m.attrs.get("GRIB_missingValue", 3.4028234663852886e+38)
    t2m   = t2m.where(t2m < fill * 0.9999)
    t2m   = t2m - 273.15                          # K → ℃
    t2m   = t2m.resample(time="1D").mean()        # 4次/天 → 日均
    ds.close()
    return t2m

print("读取数据中...")
all_years = []
for yr in range(1961, 2026):
    fpath = os.path.join(nc_dir, f"era5_2mtemp_{yr}.nc")
    if not os.path.exists(fpath):
        print(f"  ⚠️  缺失：{yr}，跳过")
        continue
    all_years.append(load_year(yr, nc_dir))
    print(f"  ✅ {yr}")

t2m_all = xr.concat(all_years, dim="time")
print(f"合并完成，共 {t2m_all.sizes['time']} 天")

# ----------------------------
# 3. Boundary mask
# ----------------------------
gdf = gpd.read_file(shp_path)
if gdf.crs is None:
    raise ValueError("Shapefile has no CRS.")
gdf  = gdf.to_crs("EPSG:4326")
poly = unary_union(gdf.geometry)

lons = t2m_all["lon"].values
lats = t2m_all["lat"].values

lon_edges = np.zeros(len(lons) + 1)
lat_edges = np.zeros(len(lats) + 1)
lon_edges[1:-1] = (lons[:-1] + lons[1:]) / 2
lat_edges[1:-1] = (lats[:-1] + lats[1:]) / 2
lon_edges[0]    = lons[0]  - (lons[1]  - lons[0])  / 2
lon_edges[-1]   = lons[-1] + (lons[-1] - lons[-2]) / 2
lat_edges[0]    = lats[0]  - (lats[1]  - lats[0])  / 2
lat_edges[-1]   = lats[-1] + (lats[-1] - lats[-2]) / 2

inside_mask = np.zeros((len(lats), len(lons)), dtype=bool)
for i in range(len(lats)):
    for j in range(len(lons)):
        cell_poly = box(lon_edges[j], lat_edges[i],
                        lon_edges[j+1], lat_edges[i+1])
        if cell_poly.intersects(poly):
            inside_mask[i, j] = True

inside_mask_da = xr.DataArray(
    inside_mask,
    coords={"lat": lats, "lon": lons},
    dims=("lat", "lon")
)

# ----------------------------
# 4. MAM selection
# ----------------------------
def select_mam(da):
    return da.sel(time=da["time"].dt.month.isin([3, 4, 5]))

t2m_mam  = select_mam(t2m_all)
t2m_2025 = t2m_mam.sel(time=slice("2025-03-01", "2025-05-31"))
t2m_clim = t2m_mam.sel(time=slice("1991-03-01", "2020-05-31"))

# ----------------------------
# 5. Daily climatology aligned to 2025
# ----------------------------
t2m_clim      = t2m_clim.assign_coords(md=t2m_clim["time"].dt.strftime("%m-%d"))
clim_daily    = t2m_clim.groupby("md").mean("time", skipna=True)
t2m_2025      = t2m_2025.assign_coords(md=t2m_2025["time"].dt.strftime("%m-%d"))
clim_for_2025 = clim_daily.sel(md=t2m_2025["md"])

# ----------------------------
# 6. Regional means
# ----------------------------
anom_2025   = t2m_2025 - clim_for_2025
anom_region = anom_2025.where(inside_mask_da).mean(dim=("lat", "lon"), skipna=True)

t2m2025_region = t2m_2025.where(inside_mask_da).mean(dim=("lat", "lon"), skipna=True)
clim_region    = clim_for_2025.where(inside_mask_da).mean(dim=("lat", "lon"), skipna=True)

dates        = anom_region["time"].values
anom_vals    = np.asarray(anom_region.values).squeeze()
t2m2025_vals = np.asarray(t2m2025_region.values).squeeze()
clim_vals    = np.asarray(clim_region.values).squeeze()

# ----------------------------
# 7. 轴域计算
# ----------------------------
finite = np.isfinite(anom_vals)
a_min  = np.nanmin(anom_vals[finite])
a_max  = np.nanmax(anom_vals[finite])
a_span = a_max - a_min if a_max > a_min else 1.0

t_max  = np.nanmax(np.concatenate([
    t2m2025_vals[np.isfinite(t2m2025_vals)],
    clim_vals[np.isfinite(clim_vals)]
]))
t_min  = np.nanmin(np.concatenate([
    t2m2025_vals[np.isfinite(t2m2025_vals)],
    clim_vals[np.isfinite(clim_vals)]
]))

# 左轴：零线在图面约 45% 高度处（与降水模板逻辑完全一致）
a_top       = a_max + 0.10 * a_span
left_bottom = -a_top * (0.45 / 0.55)
left_top    = a_top

# 右轴：气温有负值，上下各留 10% 余量
t_range     = t_max - t_min
right_bottom = t_min - 0.10 * t_range
right_top    = t_max + 0.15 * t_range

# ----------------------------
# 8. Plot
# ----------------------------
golden = 1.618
fig_w  = 5.5
fig_h  = fig_w / golden

fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=150)
fig.subplots_adjust(top=0.78, bottom=0.14, left=0.12, right=0.88)

# ── 色彩（与降水模板对称）────────────────────────────────────
pos_red   = "#F4A582"    # 暖异常：橙红
neg_blue  = "#92C5DE"    # 冷异常：蓝
line_2025 = "#C0392B"    # 中国红
line_clim = "#7B68C8"    # 蓝紫

colors = np.where(anom_vals >= 0, pos_red, neg_blue)

# ── 左轴：柱状图（异常）──────────────────────────────────────
ax.bar(
    dates, anom_vals,
    color=colors,
    width=np.timedelta64(22, "h"),
    linewidth=0.2,
    edgecolor="white",
    alpha=0.75,
    zorder=2,
)
ax.axhline(0, color="0.20", linewidth=0.9, zorder=4)
ax.yaxis.grid(False)
ax.xaxis.grid(False)

ax.set_ylim(left_bottom, left_top)
ax.set_ylabel("Anomaly (°C)", fontsize=9, fontfamily="Times New Roman")
ax.set_xlabel("Date (2025)",  fontsize=9, fontfamily="Times New Roman")

ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=5, integer=False))

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.7)
    spine.set_color("0.25")

# ── 右轴：气温曲线 ────────────────────────────────────────────
ax2 = ax.twinx()
ax2.set_ylim(right_bottom, right_top)

line_obs, = ax2.plot(
    dates, t2m2025_vals,
    color=line_2025, linewidth=1.3,
    linestyle="-", alpha=0.92,
    zorder=5, label="2025 daily T2m"
)
line_cli, = ax2.plot(
    dates, clim_vals,
    color=line_clim, linewidth=1.3,
    linestyle="-", alpha=0.90,
    zorder=5, label="Climatology (1991–2020)"
)

ax2.set_ylabel("2m Temperature (°C)",
               fontsize=9, fontfamily="Times New Roman", labelpad=6)
ax2.yaxis.set_major_locator(mticker.MaxNLocator(nbins=4, integer=False))
ax2.tick_params(axis="y", labelsize=8, width=0.7, length=3, colors="0.25")
for label in ax2.get_yticklabels():
    label.set_fontfamily("Times New Roman")

for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.7)
    spine.set_color("0.25")

# ── X 轴 ──────────────────────────────────────────────────────
start = np.datetime64("2025-03-01")
end   = np.datetime64("2025-05-31")

tick_dates = np.array([
    "2025-03-01", "2025-03-15",
    "2025-04-01", "2025-04-15",
    "2025-05-01", "2025-05-15",
    "2025-05-31"
], dtype="datetime64[D]")
tick_dates = tick_dates[(tick_dates >= start) & (tick_dates <= end)]

ax.set_xlim(start, end)
ax.set_xticks(tick_dates)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
ax.tick_params(axis="x", which="major", length=3, width=0.7, labelsize=8, pad=2)
ax.tick_params(axis="y", labelsize=8, width=0.7, length=3)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontfamily("Times New Roman")

ax.axvline(start, color="0.75", linewidth=0.6, zorder=0)
ax.axvline(end,   color="0.75", linewidth=0.6, zorder=0)
ax.margins(x=0)

# ── 图例：图框顶部外侧 ────────────────────────────────────────
legend_handles = [
    Patch(facecolor=pos_red,  edgecolor="0.55", linewidth=0.4,
          alpha=0.75, label="Warm anomaly"),
    Patch(facecolor=neg_blue, edgecolor="0.55", linewidth=0.4,
          alpha=0.75, label="Cold anomaly"),
    line_obs,
    line_cli,
]
ax.legend(
    handles=legend_handles,
    loc="lower left",
    bbox_to_anchor=(0.0, 1.02),
    ncol=2,
    fontsize=7,
    framealpha=0.0,
    edgecolor="none",
    handlelength=1.6,
    handletextpad=0.45,
    columnspacing=0.8,
    borderpad=0.3,
    labelspacing=0.3,
    prop={"family": "Times New Roman", "size": 7},
)

fig.savefig(out_pdf, format="pdf", dpi=300, bbox_inches="tight", metadata=None)
plt.show()
print("PDF saved to:", out_pdf)
